In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================

!pip install --quiet huggingface_hub requests python-dotenv pandas numpy
!pip install --quiet snowflake-connector-python "snowflake-connector-python[pandas]"
!pip install --quiet kaggle qdrant-client sentence-transformers
!pip install --quiet transformers accelerate pillow qwen-vl-utils

print("All dependencies installed")

In [ ]:
# ============================================================
# Cell 2: Configure Kaggle API Key
# ============================================================

from google.colab import files
import os

files.upload()  # Upload kaggle.json

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
os.rename("kaggle.json", os.path.join(kaggle_dir, "kaggle.json"))
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

print("Kaggle API key configured")

In [ ]:
# ============================================================
# Cell 3: Download Datasets
# ============================================================

from huggingface_hub import snapshot_download

# Jewelry dataset from HuggingFace
snapshot_download(
    repo_id="sidd707/jewelry-design-dataset",
    repo_type="dataset",
    local_dir="/content/jewelry_hf"
)
print("Jewelry dataset downloaded")

# Property and Car datasets from Kaggle
!kaggle datasets download -d sudhanshu2198/ripik-hackfest --unzip -p /content/cars
!kaggle datasets download -d mikhailma/house-rooms-streets-image-dataset --unzip -p /content/property

print("All datasets downloaded")

In [ ]:
# ============================================================
# Cell 4: Extract Jewelry ZIP
# ============================================================

import zipfile
import os

zip_path = '/content/jewelry_hf/dataset.zip'
extract_path = '/content/jewelry_hf/extracted'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Jewelry ZIP extracted")
for item in os.listdir(extract_path):
    full_path = os.path.join(extract_path, item)
    print(f"  {item}:", os.listdir(full_path)[:5] if os.path.isdir(full_path) else "")

In [ ]:
# ============================================================
# Cell 5: Shared Utilities
# ============================================================

import os
import uuid
import json
import pandas as pd

def get_value_tier(price_str):
    """Map a price string to a value tier: high / medium / low / unknown."""
    if price_str is None:
        return "unknown"
    try:
        price = float(str(price_str).replace('$', '').replace(',', '').strip())
        if price >= 5000:
            return "high"
        elif price >= 500:
            return "medium"
        else:
            return "low"
    except (ValueError, TypeError):
        return "unknown"

In [ ]:
# ============================================================
# Cell 6: Build Jewelry Records
# ============================================================

BASE_PATH = '/content/jewelry_hf/extracted/dataset'

JEWELRY_VALUE_MAP = {
    'ring_best':    'high',
    'bracelet':     'medium',
    'necklace':     'high',
    'earring_best': 'medium',
}

labels = pd.read_csv(os.path.join(BASE_PATH, 'dataset_labels.csv'), encoding='latin-1')

jewelry_records = []
for _, row in labels.iterrows():
    raw_path = str(row['image_path']).replace('\\', '/').replace('$', '/')
    category = raw_path.split('/')[0]

    jewelry_records.append({
        "record_id":            str(uuid.uuid4()),
        "image_id":             os.path.basename(raw_path),
        "image_path":           os.path.join(BASE_PATH, raw_path),
        "object_type":          "jewelry",
        "object_name":          str(row['description'])[:100],
        "condition":            "new",
        "estimated_value_tier": JEWELRY_VALUE_MAP.get(category, 'medium'),
        "location_context":     "home interior",
        "source_dataset":       "jewelry_hf",
        "source_flag":          "GROUND_TRUTH",
    })

jewel_clean = pd.DataFrame(jewelry_records)
print(f"Jewelry: {len(jewel_clean)} records")
print(jewel_clean['object_name'].head(3))

In [ ]:
# ============================================================
# Cell 7: Build Property Records
# ============================================================

PROPERTY_BASE = '/content/property/kaggle_room_street_data'

# Map filename prefix to object name and location context
ROOM_MAP = {
    'bed':        ('bedroom',       'home interior',  'medium'),
    'bath':       ('bathroom',      'home interior',  'medium'),
    'kitchen':    ('kitchen',       'home interior',  'medium'),
    'living':     ('living room',   'home interior',  'medium'),
    'din':        ('dining room',   'home interior',  'medium'),
    'kid':        ('kids room',     'home interior',  'low'),
    'living_':    ('living room',   'home interior',  'medium'),
}

STREET_MAP = {
    'retail':       ('retail property',      'outdoor', 'high'),
    'industrial':   ('industrial property',  'outdoor', 'medium'),
    'apartments':   ('apartment building',   'outdoor', 'high'),
    'church':       ('church',               'outdoor', 'medium'),
    'portland':     ('residential property', 'outdoor', 'high'),
    'house':        ('house exterior',       'outdoor', 'high'),
    'street':       ('street property',      'outdoor', 'medium'),
}

def get_property_meta(filename, folder_type):
    """Derive object_name, location_context, value_tier from filename prefix."""
    name_lower = filename.lower()

    if folder_type == 'house_data':
        for prefix, (obj_name, location, value) in ROOM_MAP.items():
            if name_lower.startswith(prefix):
                return obj_name, location, value
        return 'home interior', 'home interior', 'medium'

    elif folder_type == 'street_data':
        for prefix, (obj_name, location, value) in STREET_MAP.items():
            if prefix in name_lower:
                return obj_name, location, value
        return 'property exterior', 'outdoor', 'medium'

    return 'property', 'unknown', 'medium'

property_records = []
skipped = 0

for folder_type in ['house_data', 'street_data']:
    folder_path = f'{PROPERTY_BASE}/{folder_type}'
    if not os.path.isdir(folder_path):
        print(f"  Skipping {folder_type} — not found")
        continue

    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            skipped += 1
            continue

        obj_name, location, value_tier = get_property_meta(filename, folder_type)

        property_records.append({
            "record_id":            str(uuid.uuid4()),
            "image_id":             filename,
            "image_path":           os.path.join(folder_path, filename),
            "object_type":          "property",
            "object_name":          obj_name,
            "condition":            "good",
            "estimated_value_tier": value_tier,
            "location_context":     location,
            "source_dataset":       "house_rooms_streets",
            "source_flag":          "GROUND_TRUTH",
        })

property_clean = pd.DataFrame(property_records)
print(f"   Property: {len(property_clean)} records")
print(f"   Skipped:  {skipped}")
print(f"\nBreakdown by object name:")
print(property_clean['object_name'].value_counts())
print(f"\nBreakdown by location:")
print(property_clean['location_context'].value_counts())

In [ ]:
# ============================================================
# Cell 8: Build Car Records
# ============================================================

CARS_IMAGE_FOLDER = '/content/cars/train/train/images'

CAR_LABEL_MAP = {
    1: "no damage",
    2: "minor damage",
    3: "moderate damage",
    4: "severe damage",
}

CAR_CONDITION_MAP = {
    1: "excellent",
    2: "good",
    3: "fair",
    4: "poor",
}

cars_df = pd.read_csv('/content/cars/train/train/train.csv')

car_records = []
for _, row in cars_df.iterrows():
    label_num = int(row.get('label', 1))
    filename  = str(row.get('filename', ''))

    car_records.append({
        "record_id":            str(uuid.uuid4()),
        "image_id":             filename,
        "image_path":           os.path.join(CARS_IMAGE_FOLDER, filename),
        "object_type":          "vehicle",
        "object_name":          CAR_LABEL_MAP.get(label_num, 'vehicle'),
        "condition":            CAR_CONDITION_MAP.get(label_num, 'unknown'),
        "estimated_value_tier": "medium",
        "location_context":     "outdoor",
        "source_dataset":       "ripik_cars",
        "source_flag":          "GROUND_TRUTH",
    })

cars_clean = pd.DataFrame(car_records)
print(f"Cars: {len(cars_clean)} records")
print(cars_clean['condition'].value_counts())

In [ ]:
# ============================================================
# Cell 9: Combine All Records
# ============================================================

SCHEMA_COLS = [
    'record_id', 'image_id', 'image_path', 'object_type', 'object_name',
    'condition', 'estimated_value_tier', 'location_context',
    'source_dataset', 'source_flag'
]

ground_truth = pd.concat([jewel_clean, property_clean, cars_clean], ignore_index=True)[SCHEMA_COLS]

print(f"Total ground truth records: {len(ground_truth)}")
print("\nBy source dataset:")
print(ground_truth['source_dataset'].value_counts())
print("\nBy object type:")
print(ground_truth['object_type'].value_counts())

In [ ]:
# ============================================================
# Cell 10: Load into Snowflake
# ============================================================

import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from google.colab import userdata

conn = snowflake.connector.connect(
    account=userdata.get('SNOWFLAKE_ACCOUNT'),
    user=userdata.get('SNOWFLAKE_USER'),
    password=userdata.get('SNOWFLAKE_PASSWORD'),
    warehouse=userdata.get('SNOWFLAKE_WAREHOUSE'),
    database=userdata.get('SNOWFLAKE_DATABASE'),
    schema=userdata.get('SNOWFLAKE_SCHEMA'),
    role=userdata.get('SNOWFLAKE_ROLE')
)

cursor = conn.cursor()

# Truncate existing data
cursor.execute("SELECT COUNT(*) FROM GROUND_TRUTH")
count = cursor.fetchone()[0]
print(f"Existing records: {count} — truncating")
cursor.execute("TRUNCATE TABLE GROUND_TRUTH")

# Uppercase columns for Snowflake
ground_truth.columns = [col.upper() for col in ground_truth.columns]

# Bulk insert
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=ground_truth,
    table_name='GROUND_TRUTH',
    auto_create_table=False
)

print(f"{nrows} records inserted into Snowflake")

# Restore lowercase
ground_truth.columns = [col.lower() for col in ground_truth.columns]

# Verify
cursor.execute("SELECT object_type, COUNT(*) FROM GROUND_TRUTH GROUP BY object_type ORDER BY COUNT(*) DESC")
print(f"\nFinal breakdown in Snowflake:")
for row in cursor.fetchall():
    print(f"  {row[0]:12s}: {row[1]}")

cursor.close()
conn.close()

In [ ]:
# ============================================================
# Cell 11: Load Qwen2-VL-7B-Instruct
# ============================================================

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_ID  = "Qwen/Qwen2-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model     = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()
print(f"{MODEL_ID} loaded")

In [ ]:
# ============================================================
#  Cell 12: Qdrant: Index Ground Truth Records
# ============================================================

from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct,
    Filter, FieldCondition, MatchValue, PayloadSchemaType
)
from sentence_transformers import SentenceTransformer

QDRANT_URL        = userdata.get('QDRANT_URL')
QDRANT_API_KEY    = userdata.get('QDRANT_API_KEY')
QDRANT_COLLECTION = "object_labels"

qdrant        = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embed_model   = SentenceTransformer("all-MiniLM-L6-v2")

# Create collection if needed
# Truncate existing Qdrant collection and re-index
if qdrant.collection_exists(QDRANT_COLLECTION):
    qdrant.delete_collection(QDRANT_COLLECTION)
    print(f" Deleted old collection '{QDRANT_COLLECTION}'")

qdrant.create_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)
qdrant.create_payload_index(
    collection_name=QDRANT_COLLECTION,
    field_name="object_type",
    field_schema=PayloadSchemaType.KEYWORD,
)
print(f" Collection '{QDRANT_COLLECTION}' recreated")

# Re-embed and upsert all records
gt_lower = ground_truth.copy()
gt_lower.columns = [c.lower() for c in gt_lower.columns]

BATCH_SIZE     = 100
rows           = gt_lower.to_dict(orient="records")
total_upserted = 0

for i in range(0, len(rows), BATCH_SIZE):
    batch   = rows[i : i + BATCH_SIZE]
    texts   = [record_to_text(r) for r in batch]
    vectors = embed_model.encode(texts, show_progress_bar=False).tolist()

    points = [
        PointStruct(
            id      = str(uuid.uuid4()),
            vector  = vectors[j],
            payload = {
                "record_id":            batch[j]["record_id"],
                "image_id":             batch[j]["image_id"],
                "object_type":          batch[j]["object_type"],
                "object_name":          batch[j]["object_name"],
                "condition":            batch[j]["condition"],
                "estimated_value_tier": batch[j]["estimated_value_tier"],
                "location_context":     batch[j]["location_context"],
                "source_dataset":       batch[j]["source_dataset"],
            }
        )
        for j in range(len(batch))
    ]
    qdrant.upsert(collection_name=QDRANT_COLLECTION, points=points)
    total_upserted += len(points)
    print(f"  Upserted {total_upserted}/{len(rows)}", end="\r")

print(f"\n {total_upserted} records indexed into Qdrant")

# Verify counts per object type
for obj_type in ['property', 'vehicle', 'jewelry']:
    result = qdrant.count(
        collection_name=QDRANT_COLLECTION,
        count_filter=Filter(must=[
            FieldCondition(key="object_type", match=MatchValue(value=obj_type))
        ])
    )
    print(f"  {obj_type:12s}: {result.count} vectors")

In [ ]:
# ============================================================
# Cell 13: RAG Retrieval Function
# ============================================================

def retrieve_context(query: str, object_type: str = None, top_k: int = 3) -> str:
    """
    Embed query → search Qdrant → return formatted context string
    for injection into the Qwen2-VL prompt.
    """
    query_vector  = embed_model.encode(query).tolist()

    search_filter = None
    if object_type:
        search_filter = Filter(must=[
            FieldCondition(key="object_type", match=MatchValue(value=object_type))
        ])

    results = qdrant.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
        query_filter=search_filter,
    )

    if not results.points:
        return ""

    lines = []
    for i, hit in enumerate(results.points, 1):
        p = hit.payload
        lines.append(
            f"Example {i}: object_type={p['object_type']}, "
            f"object_name={p['object_name']}, condition={p['condition']}, "
            f"value_tier={p['estimated_value_tier']}, "
            f"location={p['location_context']}"
        )
    return "\n".join(lines)

In [ ]:
# ============================================================
# Cell 14: Prompt Builder
# ============================================================

# Per-type label schemas so the model knows exactly what fields matter
TYPE_SCHEMAS = {
    "vehicle": {
        "damage_type":  "describe damage or 'none'",
        "severity":     "minor | moderate | severe | none",
        "vehicle_part": "damaged part e.g. 'front bumper', or 'none'",
    },
    "jewelry": {
        "jewelry_type": "ring | necklace | bracelet | earring | other",
        "material":     "gold | silver | platinum | mixed | unknown",
        "gemstone":     "diamond | ruby | sapphire | emerald | none | unknown",
    },
    "property": {
    "room_type":   "bedroom | bathroom | kitchen | living room | dining room | exterior | other",
    "damage_type": "describe damage or 'none'",
    "severity":    "minor | moderate | severe | none",
    },
}

SYSTEM_PROMPT = (
    "You are a synthetic data generation assistant that analyzes product images.\n"
    "You output ONLY valid JSON. No explanation, no markdown, no code fences.\n"
    "Your response must be parseable by json.loads() directly."
)

def build_prompt(image, object_type: str, retrieved_context: str) -> list[dict]:
    """
    Build Qwen2-VL messages with RAG context and a type-specific label schema.
    Mirrors build_prompt() from the text model exactly — image replaces text input.
    """
    # Base fields every object type must produce
    base_schema = {
        "object_type":          "vehicle | jewelry | property",
        "object_name":          "specific name of the object",
        "condition":            "new | like-new | good | fair | poor | unknown",
        "estimated_value_tier": "high | medium | low | unknown",
        "location_context":     "indoor/studio | outdoor | home interior | showroom | unknown",
        "confidence_score":     "float 0.0 to 1.0",
    }

    # Merge in type-specific fields
    extra = TYPE_SCHEMAS.get(object_type, {})
    full_schema = {**base_schema, **extra}
    schema_str  = json.dumps(full_schema, indent=2)

    rag_block = ""
    if retrieved_context:
        rag_block = (
            "\nRetrieved reference examples — use these to ground your output:\n"
            "---\n"
            f"{retrieved_context}\n"
            "---\n"
        )

    user_text = (
        f"Analyze this image and generate synthetic metadata labels.\n"
        f"Object type hint: {object_type}\n"
        f"{rag_block}\n"
        f"Return EXACTLY this JSON structure — nothing outside it:\n"
        f'{{\n'
        f'  "image_id": "img_001",\n'
        f'  "labels": {schema_str}\n'
        f'}}\n\n'
        f"Start your response with {{ and end with }}."
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": user_text},
            ],
        },
    ]

In [ ]:
# ============================================================
# Cell 15: Parse & Validate  (mirrors text model logic)
# ============================================================

import re

VALID_CONDITIONS  = {"new", "like-new", "good", "fair", "poor", "unknown"}
VALID_VALUE_TIERS = {"high", "medium", "low", "unknown"}
VALID_SEVERITIES  = {"minor", "moderate", "severe", "none"}

class ParseError(Exception):      pass
class ValidationError(Exception): pass


def parse_output(raw: str) -> dict:
    """Strip fences, extract first valid JSON object — same as text model."""
    text = raw.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-z]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text.strip())

    brace_stack, start = 0, None
    for i, ch in enumerate(text):
        if ch == "{":
            if brace_stack == 0: start = i
            brace_stack += 1
        elif ch == "}":
            brace_stack -= 1
            if brace_stack == 0 and start is not None:
                try:    return json.loads(text[start:i+1])
                except: continue

    raise ParseError(f"No valid JSON found.\nRaw (first 300 chars): {raw[:300]}")


OBJECT_TYPE_NORMALIZER = {
    "vehicle":    "vehicle",
    "car":        "vehicle",
    "automobile": "vehicle",
    "jewelry":    "jewelry",
    "jewellery":  "jewelry",
    "ring":       "jewelry",
    "necklace":   "jewelry",
    "property":   "property",
    "house":      "property",
    "home":       "property",
    "roof":       "property",
}

def validate_and_repair(data: dict, object_type: str) -> dict:
    """
    Repair missing/invalid fields with safe defaults.
    Mirrors validate_records() from the text model.
    """
    labels = data.get("labels", {})

    # Clamp confidence
    try:
        labels["confidence_score"] = max(0.0, min(1.0, float(labels.get("confidence_score", 0.5))))
    except:
        labels["confidence_score"] = 0.5

    raw_model_type = str(labels.get("object_type", "")).lower().strip()
    normalized     = OBJECT_TYPE_NORMALIZER.get(raw_model_type, object_type)

    labels["object_type"] = object_type

    # Safe defaults for base fields
    defaults = {
        "object_type":          object_type,
        "object_name":          "unknown",
        "condition":            "unknown",
        "estimated_value_tier": "unknown",
        "location_context":     "unknown",
        "confidence_score":     0.5,
    }

    # Safe defaults for type-specific fields
    type_defaults = {
    "vehicle":  {"damage_type": "none", "severity": "none", "vehicle_part": "none"},
    "jewelry":  {"jewelry_type": "unknown", "material": "unknown", "gemstone": "none"},
    "property": {"room_type": "unknown", "damage_type": "none", "severity": "none"},
    }

    defaults.update(type_defaults.get(object_type, {}))

    for k, v in defaults.items():
        if k not in labels or labels[k] in [None, ""]:
            labels[k] = v

    # Normalize enums
    if labels.get("condition") not in VALID_CONDITIONS:
        labels["condition"] = "unknown"
    if labels.get("estimated_value_tier") not in VALID_VALUE_TIERS:
        labels["estimated_value_tier"] = "unknown"
    if object_type == "vehicle" and labels.get("severity") not in VALID_SEVERITIES:
        labels["severity"] = "none"

    data["labels"] = labels
    return data

In [ ]:
# ============================================================
# Cell 16: Core Inference Function
# ============================================================

from PIL import Image as PILImage

def generate_labels_from_image(image_path: str, object_type: str, image_id: str) -> dict:
    """
    Main function — mirrors generate_synthetic_data() from the text model.

    1. Load image from local path
    2. Retrieve RAG context from Qdrant (filtered by object_type)
    3. Build prompt with RAG context injected
    4. Run Qwen2-VL
    5. Parse + validate JSON
    6. Return structured result dict
    """
    # 1. Load image
    image = PILImage.open(image_path).convert("RGB")

    # 2. RAG retrieval — filtered to same object type for relevant examples
    query             = f"{object_type} condition value damage location"
    retrieved_context = retrieve_context(query=query, object_type=object_type, top_k=3)

    # 3. Build prompt
    messages = build_prompt(image, object_type, retrieved_context)

    # 4. Run Qwen2-VL
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=300)

    trimmed    = generated_ids[:, inputs.input_ids.shape[1]:]
    raw_output = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    # 5. Parse + validate
    try:
        parsed    = parse_output(raw_output)
        validated = validate_and_repair(parsed, object_type)
        parse_ok  = True
    except ParseError:
        validated = {"labels": {}}
        validate_and_repair(validated, object_type)
        parse_ok  = False

    # 6. Attach lineage
    validated["image_id"]         = image_id
    validated["inference_id"]     = str(uuid.uuid4())
    validated["rag_context_used"] = retrieved_context[:500]
    validated["raw_model_output"] = raw_output[:2000]
    validated["parse_success"]    = parse_ok

    return validated


# ============================================================
# Snowflake Output Table + Save Helper
# ============================================================

def get_sf_conn():
    return snowflake.connector.connect(
        account   = userdata.get('SNOWFLAKE_ACCOUNT'),
        user      = userdata.get('SNOWFLAKE_USER'),
        password  = userdata.get('SNOWFLAKE_PASSWORD'),
        warehouse = userdata.get('SNOWFLAKE_WAREHOUSE'),
        database  = userdata.get('SNOWFLAKE_DATABASE'),
        schema    = userdata.get('SNOWFLAKE_SCHEMA'),
        role      = userdata.get('SNOWFLAKE_ROLE'),
    )

def ensure_output_table():
    conn   = get_sf_conn()
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS SYNTHETIC_LABELS (
            inference_id          VARCHAR,
            image_id              VARCHAR,
            source_dataset        VARCHAR,
            -- base labels (all types)
            object_type           VARCHAR,
            object_name           VARCHAR,
            condition             VARCHAR,
            estimated_value_tier  VARCHAR,
            location_context      VARCHAR,
            confidence_score      FLOAT,
            -- vehicle-specific
            damage_type           VARCHAR,
            severity              VARCHAR,
            vehicle_part          VARCHAR,
            -- jewelry-specific
            jewelry_type          VARCHAR,
            material              VARCHAR,
            gemstone              VARCHAR,
            -- property-specific
            room_type             VARCHAR,
            -- lineage
            parse_success         BOOLEAN,
            rag_context_used      VARCHAR(1000),
            raw_model_output      VARCHAR(4000),
            created_at            TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cursor.close()
    conn.close()
    print("SYNTHETIC_LABELS table ready")

ensure_output_table()

# Collect results in memory, write once at the end
results_buffer = []

def save_to_buffer(result: dict, source_dataset: str):
    labels = result.get("labels", {})
    results_buffer.append({
        "INFERENCE_ID":          result.get("inference_id"),
        "IMAGE_ID":              result.get("image_id"),
        "SOURCE_DATASET":        source_dataset,
        "OBJECT_TYPE":           labels.get("object_type"),
        "OBJECT_NAME":           labels.get("object_name"),
        "CONDITION":             labels.get("condition"),
        "ESTIMATED_VALUE_TIER":  labels.get("estimated_value_tier"),
        "LOCATION_CONTEXT":      labels.get("location_context"),
        "CONFIDENCE_SCORE":      labels.get("confidence_score"),
        "DAMAGE_TYPE":           labels.get("damage_type"),
        "SEVERITY":              labels.get("severity"),
        "VEHICLE_PART":          labels.get("vehicle_part"),
        "JEWELRY_TYPE":          labels.get("jewelry_type"),
        "MATERIAL":              labels.get("material"),
        "GEMSTONE":              labels.get("gemstone"),
        "ROOM_TYPE":             labels.get("room_type"),
        "PARSE_SUCCESS":         result.get("parse_success", False),
        "RAG_CONTEXT_USED":      result.get("rag_context_used", "")[:1000],
        "RAW_MODEL_OUTPUT":      result.get("raw_model_output", "")[:4000],
    })

def flush_buffer_to_snowflake():
    if not results_buffer:
        print("Nothing to flush")
        return
    df   = pd.DataFrame(results_buffer)
    conn = get_sf_conn()
    write_pandas(conn=conn, df=df, table_name="SYNTHETIC_LABELS", auto_create_table=False)
    conn.close()
    print(f"Flushed {len(results_buffer)} records to Snowflake")
    results_buffer.clear()

In [ ]:
# ============================================================
# Cell 17: Batch Pipeline: Ground Truth → VLM → Snowflake
# ============================================================

from tqdm import tqdm

def run_batch(gt_df: pd.DataFrame, max_images: int = 50, flush_every: int = 10):
    gt_lower = gt_df.copy()
    gt_lower.columns = [c.lower() for c in gt_lower.columns]

    sample  = gt_lower.sample(n=min(max_images, len(gt_lower)), random_state=42)
    success = 0
    failed  = 0

    for i, (_, row) in enumerate(tqdm(sample.iterrows(), total=len(sample), desc="Labeling images")):
        image_path = row["image_path"]

        if not image_path or pd.isna(image_path) or not os.path.exists(str(image_path)):
            print(f"  Missing: {image_path}")
            failed += 1
            continue

        try:
            result = generate_labels_from_image(
                image_path  = image_path,
                object_type = row["object_type"],
                image_id    = row["image_id"],
            )
            # Add to buffer instead of writing immediately
            save_to_buffer(result, source_dataset=row["source_dataset"])
            success += 1
            print(
                f"  {row['image_id'][:30]:30s} | "
                f"{result['labels']['object_type']:8s} | "
                f"condition={result['labels']['condition']:8s} | "
                f"conf={result['labels']['confidence_score']:.2f}"
            )
        except Exception as e:
            print(f"  {row['image_id']} — {e}")
            failed += 1

        # Flush to Snowflake every N images
        if (i + 1) % flush_every == 0:
            flush_buffer_to_snowflake()

    # Final flush
    flush_buffer_to_snowflake()
    print(f"\nBatch done: {success} succeeded, {failed} failed")


run_batch(ground_truth, max_images=100, flush_every=10)

In [ ]:
# ============================================================
# Cell 18: Verify Results in Snowflake
# ============================================================

conn   = get_sf_conn()
cursor = conn.cursor()

cursor.execute("""
    SELECT
        object_type,
        COUNT(*)                                          AS total,
        ROUND(AVG(confidence_score), 3)                  AS avg_confidence,
        SUM(CASE WHEN parse_success THEN 1 ELSE 0 END)   AS parsed_ok
    FROM SYNTHETIC_LABELS
    GROUP BY object_type
    ORDER BY total DESC
""")

rows = cursor.fetchall()
print(f"\n{'object_type':12s}  {'total':>6}  {'avg_conf':>10}  {'parsed_ok':>10}")
print(f"{'─'*12}  {'─'*6}  {'─'*10}  {'─'*10}")
for r in rows:
    print(f"{str(r[0]):12s}  {r[1]:>6}  {float(r[2] or 0):>10.3f}  {r[3]:>10}")

cursor.close()
conn.close()

In [ ]:
# ============================================================
# Cell 19: Random Check
# ============================================================

from PIL import Image as PILImage
import matplotlib.pyplot as plt

# Replace the two lines below with this to choose a specific image to run the model
# test_row = ground_truth[ground_truth["image_path"].str.contains("some_filename")].iloc[0]

# Pick any row from ground truth to test
test_row = ground_truth.sample(1).iloc[0]
test_row.index = [i.lower() for i in test_row.index]

# Show the image
img = PILImage.open(test_row["image_path"])
plt.imshow(img)
plt.axis("off")
plt.show()

# Run the model and print output
result = generate_labels_from_image(
    image_path  = test_row["image_path"],
    object_type = test_row["object_type"],
    image_id    = test_row["image_id"],
)

print(json.dumps(result["labels"], indent=2))